# Lesson 02 - 探索 Microsoft Agent 框架

**Microsoft Agent 框架 (MAF)** 是用于构建 AI 代理的统一框架。它提供了一个简洁、可组合的架构，包含四个核心构建模块：

- **Client** – 连接到 AI 模型端点并处理通信
- **Agent** – 用指令和工具定义包装客户端
- **Tools** – 通过模型可调用的自定义函数扩展代理能力
- **Session** – 维护多轮交互的对话历史

在本课中，我们将使用这些概念构建一个**旅行预订代理**，用于检查目的地可用性。


## 设置


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## 理解 Agent 框架架构

Microsoft Agent 框架遵循分层架构：

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **客户端** – 一个 `AzureAIProjectAgentProvider` 连接到 Azure OpenAI 部署。它负责身份验证、请求格式化和响应解析。
2. **代理** – 通过客户端的 `provider.create_agent()` 创建的代理，将模型访问与指令（系统提示）和工具结合在一起。
3. **工具** – 用 `@tool` 装饰的 Python 函数，代理可以调用它们执行操作或检索数据。
4. **会话** – 一个 `AgentSession` 对象（通过 `agent.create_session()` 创建），用于存储对话历史，实现多轮对话，让代理记住之前的上下文。

让我们一步步构建每一层。


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## 使用 @tool 装饰器添加工具

工具让代理能够执行生成文本以外的操作。`@tool` 装饰器将普通的 Python 函数转换为代理可调用的内容。

关键点：
- 使用 `Annotated[type, "description"]` 让模型理解每个参数。
- 文档字符串成为模型看到的工具描述。
- `approval_mode="never_require"` 意味着工具自动运行，无需用户确认。


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## 使用工具创建代理

现在我们将客户端、指令和工具组合成一个代理。`instructions` 作为系统提示 —— 它们定义了代理的角色和行为。


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## 带有会话的多轮对话

`AgentSession`（通过 `agent.create_session()` 创建）会跟踪对话中的所有消息。通过将相同的会话传递给每次 `agent.run()` 调用，代理可以访问完整的对话历史记录，并且可以引用之前的消息。

我们传递 `tools=[check_destination_availability]`，这样代理就可以在每一轮调用我们的可用性检查器。


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## 摘要

在本课中，您探索了 Microsoft Agent 框架的四大支柱：

| 概念 | 您学到了什么 |
|---------|------------------|
| **客户端** | `AzureAIProjectAgentProvider` 使用基于凭据的身份验证连接到 Azure OpenAI |
| **代理** | `provider.create_agent()` 将模型连接、指令和名称打包在一起 |
| **工具** | `@tool` 装饰器暴露供代理调用的 Python 函数 |
| **会话** | `agent.create_session()` 维持多轮对话的历史记录 |

这些构建模块组合在一起，创建能够进行自然对话、调用外部函数并维护上下文的代理——为后续课程中更高级的代理模式奠定基础。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：  
本文件使用 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 进行翻译。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版本的文档应被视为权威来源。对于重要信息，建议使用专业人工翻译。对于因使用本翻译而产生的任何误解或错误解释，我们不承担任何责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
